# `forex(base, quote, start, end=None)`

Daily rates for any pair of stored currencies, from FMP via `forex.prices`. Every cell runs against prod, read-only, as `readonly_user`. Change the arguments and re-run any cell.

In [ ]:
from datetime import date, timedelta

import polars as pl
from IPython.display import Markdown

from arora_data.dates import today_et
from arora_data.forex import FxBar
from arora_data.frames import to_df
from arora_data.sync import forex


def show(rows):
    """One result or a list of results as a Markdown table."""
    rows = [r.model_dump() for r in (rows if isinstance(rows, list) else [rows])]
    if not rows:
        return Markdown("0 rows")
    head = list(rows[0])
    lines = ["|".join(head), "|".join("---" for _ in head)]
    lines += ["|".join(str(r[c]).replace("|", "/") for c in head) for r in rows]
    return Markdown(f"{len(rows)} rows\n\n" + "\n".join(lines))

## A stored pair
FMP daily bars, returned as stored (`route` = stored).

In [ ]:
show(forex("USD", "CAD", today_et() - timedelta(days=7)))

## An inverted pair
Only USDCAD is stored, so CAD/USD is 1 / USDCAD. High and low swap. volume is FMP's tick count, the same either way round; vwap is 1 / vwap, a close approximation.

In [ ]:
show(forex("CAD", "USD", today_et() - timedelta(days=7)))

## A crossed pair
BRL/INR isn't stored, so it is usd_per(BRL) / usd_per(INR), from USDBRL and USDINR. A cross has open and close only, and skips a date either USD pair lacks.

In [ ]:
show(forex("BRL", "INR", today_et() - timedelta(days=7)))

## `forex("ALL", "ALL", start, end)`
Every pair of stored currencies once: stored pairs as stored, the rest crossed. Keep the range short: one week is about 50,000 rows.

In [ ]:
to_df(forex("ALL", "ALL", today_et() - timedelta(days=3)), FxBar)

## Full history in one call
EUR/USD from 1999, the start of FMP's history. There is no row limit, so the whole series comes back.

In [ ]:
eurusd = to_df(forex("EUR", "USD", date(1999, 1, 1)), FxBar)
print(f"{eurusd.height:,} days, {eurusd['date'].min()} to {eurusd['date'].max()}")
eurusd.select("date", "open", "high", "low", "close").tail(5)

## Year-end closes
The last close of each year, from the frame above.

In [ ]:
(
    eurusd.group_by(pl.col("date").dt.year().alias("year"), maintain_order=True)
    .agg(pl.col("date").last().alias("last_day"), pl.col("close").last())
    .tail(10)
)

## Several currencies in rupees, side by side
What one unit of each costs in INR. USD/INR is stored; the others are crossed through USD.

In [ ]:
start = today_et() - timedelta(days=14)
rows = [r for base in ["USD", "EUR", "GBP", "JPY", "CNY"] for r in forex(base, "INR", start)]
to_df(rows, FxBar).pivot(on="pair", index="date", values="close")

## An exotic cross
One Kuwaiti dinar in Lebanese pounds. Computed rates keep 12 significant digits, so large and tiny crosses both stay exact in a dataframe.

In [ ]:
show(forex("KWD", "LBP", today_et() - timedelta(days=7)))

## Bad input
A currency with no stored USD pair (gold XAU, IMF units, retired currencies), the same currency twice, or only one side "ALL" raise a clear error instead of returning nothing.

In [ ]:
for args in [("XAU", "USD"), ("EUR", "EUR"), ("ALL", "USD")]:
    try:
        forex(*args, today_et() - timedelta(days=7))
    except ValueError as err:
        print(f"{args}: {err}")